# ramai — the croupier

An AI that plays Moroccan Rami against you by **looking at your real cards on a real table**. The camera films continuously. The AI speaks aloud in French. You are its hands. Nobody sees the other's hand — total symmetry. The camera sees everything, so nobody can cheat without being seen.

**Author:** Amine Harch El Korane  
**Repo:** https://github.com/Vitalcheffe/ramai  
**Tests:** 149 passing

---

## How it works

1. **Print the calibration sheet** (Cell 2 generates the PDF) and put it on the table.
2. **Start the camera stream** — it's displayed live in the notebook output, 4 FPS.
3. **Calibrate** — film the sheet, the AI finds the 5 zones (MONTRE, ZONE IA, TALON, DEFAUSSE, CENTRE).
4. **Setup AI's hand** — show 14 cards one by one in MONTRE. AI remembers each by position.
5. **Play** — the AI speaks, you execute physically, the camera verifies every move.

## Target hardware

iPad on Safari, Colab app. If `getUserMedia` is blocked (Safari in iframes), the notebook falls back to `<input type=file capture=environment>` — native camera app opens per capture.

---

Run cells in order.

## Cell 1 — Install, imports, start stream + voice

In [ ]:
!pip install -q ultralytics ipywidgets gtts 2>&1 | tail -3

import os, sys, json, time, random
from pathlib import Path
from IPython.display import display, HTML, Audio, clear_output
import ipywidgets as widgets
import numpy as np
import cv2

if not Path('/content/ramai').exists():
    !git clone -q https://github.com/VitalCheffe/ramai.git /content/ramai
REPO = Path('/content/ramai')
sys.path.insert(0, str(REPO))

from rami.config import RamiConfig
from rami.cards import Card, build_deck, Hand, SUIT_SYMBOLS, RANK_NAMES
from rami.engine import (is_valid_meld, valid_melds, deadwood_score,
                          best_meld_partition, meld_points)
from rami.game import new_game, legal_moves, apply_move, Move, GameState
from rami.extensions import (designate_jokers, find_meld_extensions,
                              all_laid_melds, JokerDesignation)
from rami.counting import CardCountingState
from rami.zones import (ZoneName, ZoneMap, calibrate_zones_from_image,
                          which_zone, cards_in_zone, slot_number_for_detection,
                          draw_zones_overlay)
from rami.voice import Voice
from rami.ai.discovery import DiscoveryAI
from rami.ai.strategy import StrategyAI
from rami.ai.champion import ChampionAI
from rami.vision import (
    CardDetector, MockDetector, calibrate_camera,
    detect_discard_pile, detect_meld_clusters, find_extendable_melds,
    start_stream, stop_stream, get_latest_frame, get_frame_number,
    get_stream_status, is_streaming, capture_photo,
    prewarm_camera, init_file_input_mode, capture_photo_file_input,
    try_download_pretrained, get_model_info, get_download_url,
)

print('✓ ramai loaded — croupier mode')
print(f'  Tests: 149 (engine 79 + protocol 40 + camera/manual 12 + croupier 18)')
print()

# Start voice (silent at init — will speak when game starts)
voice = Voice(lang='fr-FR')
globals()['VOICE'] = voice
print('✓ Voice ready (French)')
print()

# Start live video stream
print('--- Starting live video stream ---')
print('A camera permission popup should appear.')
print('On iPad Safari, if no popup appears, the notebook will fall back')
print('to the native camera app (file-input mode).')
print()
stream_status = start_stream()
print(f'Stream status: {stream_status}')
if stream_status == 'streaming':
    print('  ✓ Live video is displayed above. You should see yourself.')
    voice.say('Flux vidéo actif. Je vois la table.')
elif stream_status == 'file_input':
    print('  ⚠ Live stream blocked. Falling back to native camera app.')
    init_file_input_mode()
    voice.say('Mode photo par photo. Appuie sur le bouton quand je te le demande.')
else:
    print(f'  ✗ Camera unavailable ({stream_status}).')
globals()['STREAM_STATUS'] = stream_status

## Cell 2 — Download YOLO weights + generate calibration sheet

In [ ]:
WEIGHTS = REPO / 'models' / 'yolov8n.pt'
print(f'Download URL: {get_download_url()}')
if not WEIGHTS.exists():
    print('Downloading YOLO weights...')
    result = try_download_pretrained(out_path=str(WEIGHTS), timeout=60)
    if result:
        info = get_model_info(str(WEIGHTS))
        print(f'✓ Downloaded: {info["size_mb"]} MB')
    else:
        print('✗ Download failed')
else:
    info = get_model_info(str(WEIGHTS))
    print(f'✓ Weights present: {info["size_mb"]} MB')

detector = MockDetector()
if WEIGHTS.exists():
    try:
        detector = CardDetector(weights_path=str(WEIGHTS))
        print(f'✓ Detector loaded: {len(detector.names)} classes')
    except Exception as e:
        print(f'⚠ Detector load error: {e}')
globals()['DETECTOR'] = detector

# Generate calibration sheet PDF
print()
print('--- Calibration sheet ---')
sheet_path = str(REPO / 'assets' / 'ramai_sheet.pdf')
if not Path(sheet_path).exists():
    !cd {REPO} && python scripts/make_sheet.py
print(f'✓ Sheet PDF: {sheet_path}')
print(f'  → Print this on A4. Place on table. Camera will calibrate on it.')
print(f'  → 5 zones: MONTRE, ZONE IA (15 slots), TALON, DEFAUSSE, CENTRE')

## Cell 3 — Calibrate the 5 zones (REQUIRED)

Film the printed sheet on the table. The AI finds the 5 zones. Then place a test card in MONTRE to verify detection works.

In [ ]:
calib_out = widgets.Output()
display(calib_out)

voice.say('Calibration. Filme la feuille sur la table.')

calib_btn = widgets.Button(description='📸 Calibrate zones', button_style='primary')
test_btn = widgets.Button(description='🃏 Test card in MONTRE', button_style='info')
display(widgets.HBox([calib_btn, test_btn]))

calib_state = {'zones_calibrated': False, 'test_card_detected': False, 'zone_map': None}

def on_calibrate(b):
    calib_out.clear_output()
    with calib_out:
        print('📸 Capturing frame for calibration...')
        img = capture_photo()
        if img is None:
            print('✗ Capture failed.')
            return
        zmap = calibrate_zones_from_image(img)
        if zmap is None:
            print('✗ Calibration failed.')
            return
        calib_state['zone_map'] = zmap
        calib_state['zones_calibrated'] = True
        print('✓ 5 zones located:')
        for name, box in zmap.zones.items():
            print(f'  {name.value}: ({box.x1},{box.y1}) → ({box.x2},{box.y2})')
        # Draw overlay
        overlay = draw_zones_overlay(img, zmap)
        from google.colab.patches import cv2_imshow
        cv2_imshow(overlay)
        voice.say('Calibration réussie. Pose une carte dans le carré MONTRE pour tester.')

def on_test(b):
    calib_out.clear_output()
    if not calib_state['zones_calibrated']:
        with calib_out: print('⚠ Calibrate zones first.')
        return
    with calib_out:
        print('📸 Capturing frame...')
        img = capture_photo()
        if img is None:
            print('✗ Capture failed.')
            return
        detections = DETECTOR.predict(img)
        montre_cards = cards_in_zone(detections, ZoneName.MONTRE, calib_state['zone_map'])
        print(f'Cards detected in MONTRE: {len(montre_cards)}')
        for d in montre_cards:
            print(f'  {d.rank}{d.suit} (confidence {d.confidence*100:.0f}%)')
        # Draw overlay
        overlay = draw_zones_overlay(img, calib_state['zone_map'])
        for d in detections:
            x1, y1, x2, y2 = [int(v) for v in d.bbox]
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 255), 2)
            cv2.putText(overlay, f'{d.rank}{d.suit} {d.confidence:.2f}',
                        (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        from google.colab.patches import cv2_imshow
        cv2_imshow(overlay)
        if montre_cards:
            calib_state['test_card_detected'] = True
            voice.say(f'Carte reconnue : {montre_cards[0].rank} de {montre_cards[0].suit}. Calibration validée.')
        else:
            voice.say('Aucune carte dans le carré. Réessaie.')

calib_btn.on_click(on_calibrate)
test_btn.on_click(on_test)

## Cell 4 — Configure game (AI level + variant)

In [ ]:
ai_level = widgets.Dropdown(
    options=[('Discovery (rules only, AI hand visible)', 'discovery'),
             ('Strategy (perfect card counting, AI hand hidden)', 'strategy'),
             ('Champion (RL self-play, AI hand hidden)', 'champion')],
    value='strategy', description='AI level:',
    style={'description_width': 'initial'}
)
variant = widgets.Dropdown(
    options=[('Classic Moroccan (threshold 30)', 'classic'),
             ('Rami 51 (threshold 51, no discard draw before threshold)', '51'),
             ('No threshold', 'none'),
             ('No jokers', 'nojokers')],
    value='classic', description='Variant:',
    style={'description_width': 'initial'}
)
display(ai_level, variant)

confirm = widgets.Button(description='Confirm', button_style='primary')
cfg_out = widgets.Output()
display(confirm, cfg_out)

def on_confirm(b):
    cfg_out.clear_output()
    if variant.value == 'classic':
        cfg = RamiConfig.classic_moroccan()
    elif variant.value == '51':
        cfg = RamiConfig.threshold_51()
    elif variant.value == 'none':
        cfg = RamiConfig.no_threshold()
    else:
        cfg = RamiConfig.no_jokers()
    if ai_level.value == 'discovery':
        ai = DiscoveryAI(seed=0)
    elif ai_level.value == 'strategy':
        ai = StrategyAI(seed=0)
    else:
        weights_path = str(REPO / 'models' / 'champion_weights.json')
        if not os.path.exists(weights_path):
            with cfg_out:
                print('Training Champion (500 games)...')
                !cd {REPO} && python scripts/train_champion.py --games 500 --candidates 6
        ai = ChampionAI(weights_path=weights_path, seed=0)
    with cfg_out:
        print(f'✓ Configuration confirmed')
        print(f'  AI: {ai.name}, Variant: {variant.label}')
        voice.say(f'Configuration validée. Niveau {ai.name}. Prêt à jouer.')
        globals()['CFG'] = cfg
        globals()['AI'] = ai
        globals()['AI_LEVEL'] = ai_level.value

confirm.on_click(on_confirm)

## Cell 5 — Setup: AI's 14 cards (show one by one in MONTRE)

The AI's hand stays face-down in ZONE IA. To know what it holds, you show its 14 cards ONE BY ONE in MONTRE. For each card:
1. AI recognizes it via the camera.
2. AI tells you which position to place it face-down in ZONE IA.
3. You move the card from MONTRE to the slot. AI verifies (back of card now in slot).

In [ ]:
setup_out = widgets.Output()
display(setup_out)

# Initialize game state
CFG = globals().get('CFG', RamiConfig())
AI = globals().get('AI', StrategyAI(seed=0))
AI_LEVEL = globals().get('AI_LEVEL', 'strategy')
state = new_game(CFG, seed=int(time.time()) % 1000)
counting = CardCountingState.fresh(
    CFG, ai_player_idx=1,
    ai_hand=state.players[1].hand.cards,
    initial_discard=state.discard,
)

voice.say('Phase de setup. Je vais te demander mes 14 cartes une par une.')
voice.say(f'Ma première carte, montre-la dans le carré MONTRE.')

setup_state = {'next_card_idx': 0, 'cards_shown': []}

show_card_btn = widgets.Button(description='📸 I showed a card in MONTRE',
                                  button_style='primary')
skip_btn = widgets.Button(description='Skip setup (use random hand)', button_style='warning')
display(widgets.HBox([show_card_btn, skip_btn]))

def on_show_card(b):
    setup_out.clear_output()
    idx = setup_state['next_card_idx']
    if idx >= 14:
        with setup_out:
            print('✓ All 14 cards setup. Ready to play (Cell 6).')
            voice.say('Setup terminé. À toi de jouer.')
        return
    with setup_out:
        print(f'Card {idx+1}/14 — capturing frame...')
        img = capture_photo()
        if img is None:
            print('✗ Capture failed.')
            return
        detections = DETECTOR.predict(img)
        montre_cards = cards_in_zone(detections, ZoneName.MONTRE, calib_state['zone_map'])
        if not montre_cards:
            print('No card detected in MONTRE. Place a card and try again.')
            voice.say('Aucune carte détectée. Recommence.')
            return
        d = montre_cards[0]  # take highest confidence
        if d.confidence < 0.7:
            print(f'⚠ Low confidence ({d.confidence*100:.0f}%). Showing photo for confirmation:')
            overlay = draw_zones_overlay(img, calib_state['zone_map'])
            for det in detections:
                x1, y1, x2, y2 = [int(v) for v in det.bbox]
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 165, 255), 2)
                cv2.putText(overlay, f'{det.rank}{det.suit} {det.confidence:.2f}',
                            (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
            from google.colab.patches import cv2_imshow
            cv2_imshow(overlay)
            print(f'Detected: {d.rank}{d.suit}. If wrong, retry. If right, continue.')
            return
        # Confirm card
        setup_state['cards_shown'].append(d)
        slot = idx + 1  # slots 1-14
        print(f'✓ Card {idx+1}/14: {d.rank}{d.suit} (confidence {d.confidence*100:.0f}%)')
        print(f'  → Place it face-down in slot {slot} of ZONE IA.')
        voice.say(f'{d.rank} de {d.suit}. Range-le face cachée en position {slot}.')
        setup_state['next_card_idx'] = idx + 1
        if setup_state['next_card_idx'] < 14:
            print(f'  Next card — show in MONTRE.')
            voice.say(f'Montre-moi la carte {setup_state["next_card_idx"]+1}.')
        else:
            print(f'✓ All 14 cards shown. Ready to play (Cell 6).')
            voice.say('Setup terminé. À toi de jouer.')

def on_skip(b):
    setup_out.clear_output()
    with setup_out:
        print('⚠ Skipping setup. AI hand stays unknown to camera.')
        print('  (AI still has 14 cards in memory, just not shown via camera)')
        voice.say('Setup ignoré. On commence avec une main aléatoire.')
        setup_state['next_card_idx'] = 14  # mark as done

show_card_btn.on_click(on_show_card)
skip_btn.on_click(on_skip)

## Cell 6 — Your turn

Click the buttons to confirm physical gestures:
- **Drew from stock** — you drew a card from TALON
- **Took discard** — you took the top of DEFAUSSE
- **Laid meld** — you put cards in CENTRE
- **Discarded** — you put a card on DEFAUSSE
- **End my turn** — pass to AI

In [ ]:
human_out = widgets.Output()

draw_stock_btn = widgets.Button(description='Drew from stock', button_style='info')
draw_discard_btn = widgets.Button(description='Took discard', button_style='warning')
if CFG.block_discard_before_threshold and not state.players[0].has_laid_first:
    draw_discard_btn.disabled = True
end_turn_btn = widgets.Button(description='End my turn', button_style='success')

display(widgets.HBox([draw_stock_btn, draw_discard_btn, end_turn_btn]), human_out)

human_action = {'draw_source': None, 'photo_taken': False}

def on_draw_stock(b):
    human_out.clear_output()
    if not state.stock:
        with human_out: print('Stock is empty.')
        return
    drawn = state.stock.pop()
    state.players[0].hand.add(drawn)
    counting.record_draw(0, 'stock', drawn, ai_player_idx=1)
    human_action['draw_source'] = 'stock'
    with human_out:
        print(f'You drew from stock (camera cannot see your hand).')
        voice.say('Tu as pioché au talon.')

def on_draw_discard(b):
    human_out.clear_output()
    if not state.discard:
        with human_out: print('No discard pile.')
        return
    drawn = state.discard.pop()
    state.players[0].hand.add(drawn)
    counting.record_draw(0, 'discard', drawn, ai_player_idx=1)
    human_action['draw_source'] = 'discard'
    with human_out:
        print(f'You took the discard: {drawn.name}')
        voice.say(f'Tu as pris la défausse : {drawn.name}.')

def on_end_turn(b):
    human_out.clear_output()
    if not human_action['draw_source']:
        with human_out: print('You must draw first.')
        return
    # Mandatory photo of discard pile
    with human_out:
        print('📸 MANDATORY photo of discard pile:')
        img = capture_photo()
        if img is None:
            print('✗ Capture failed.')
            return
        # Detect top of discard
        detections = DETECTOR.predict(img)
        discard_cards = cards_in_zone(detections, ZoneName.DEFAUSSE, calib_state['zone_map'])
        if discard_cards:
            top = discard_cards[0]
            print(f'Top of discard detected: {top.rank}{top.suit} (conf {top.confidence*100:.0f}%)')
            voice.say(f'Défausse : {top.rank} de {top.suit}.')
        else:
            print('⚠ No card detected in DEFAUSSE zone. Showing photo for confirmation.')
            overlay = draw_zones_overlay(img, calib_state['zone_map'])
            from google.colab.patches import cv2_imshow
            cv2_imshow(overlay)
            voice.say('Défausse non détectée. Vérifie.')
        # Card counting check
        est = counting.opponent_hand_estimate(CFG, opponent_idx=0,
                                                stock_size=len(state.stock))
        print(f'Your hand (inferred by AI): {est["hand_count"]} cards')
        if counting.is_opponent_empty(0):
            print('🏁 You won!')
            state.winner = 0
            state.terminal = True
            return
        state.current = 1
        state.turn += 1
        voice.say('À moi.')

draw_stock_btn.on_click(on_draw_stock)
draw_discard_btn.on_click(on_draw_discard)
end_turn_btn.on_click(on_end_turn)

## Cell 7 — RAMAI's turn

The AI announces its move aloud in French. You execute the gestures physically. The camera verifies each step.

In [ ]:
ai_play_btn = widgets.Button(description='🎯 RAMAI plays', button_style='primary')
ai_out = widgets.Output()
display(ai_play_btn, ai_out)

def on_ai_play(b):
    ai_out.clear_output()
    with ai_out:
        if state.terminal:
            print('Game over.')
            return
        if state.current != 1:
            print("Not RAMAI's turn. Play your turn first (Cell 6).")
            return
        print(f'--- Turn {state.turn+1} | RAMAI ({AI.name}) ---')
        m = AI.decide(state)

        # 1. Draw announcement
        if m.draw_source == 'discard':
            voice.say(f'Je prends la défausse : {state.top_discard.name}.')
            print(f'RAMAI: "Je prends la défausse: {state.top_discard.name}."')
        else:
            voice.say('Je pioche. Montre-moi la carte dans le carré.')
            print(f'RAMAI: "Je pioche. Montre-moi la carte dans le carré MONTRE."')

        # 2. Apply draw
        if m.draw_source == 'stock':
            drawn = state.stock.pop()
            counting.record_draw(1, 'stock', drawn, ai_player_idx=1)
        else:
            drawn = state.discard.pop()
            counting.record_draw(1, 'discard', drawn, ai_player_idx=1)
        state.players[1].hand.add(drawn)

        # 3. Laydowns
        if m.laydowns:
            cards_to_show = []
            for meld in m.laydowns:
                cards_str = ' '.join(c.name for c in meld)
                voice.say(f'Je pose : {cards_str}.')
                print(f'RAMAI: "Je pose: {cards_str}."')
                # Joker designations
                desigs = designate_jokers(meld, CFG)
                for d in desigs:
                    voice.say(f'Le joker vaut {d.name}.')
                    print(f'  Joker: {d.name}')
                for card in meld:
                    state.players[1].hand.remove(card)
                state.players[1].laid_melds.append(meld)
                counting.record_meld(1, meld)
                if not state.players[1].has_laid_first:
                    state.players[1].has_laid_first = True

        # 4. Discard
        state.players[1].hand.remove(m.discard)
        state.discard.append(m.discard)
        counting.record_discard(1, m.discard)
        voice.say(f'Je jette : {m.discard.name}. Pose-la face visible sur la défausse.')
        print(f'RAMAI: "Je jette: {m.discard.name}. Pose-la sur la défausse."')

        # 5. Mandatory photo of discard
        print('📸 MANDATORY photo of new discard pile:')
        img = capture_photo()
        if img is not None:
            detections = DETECTOR.predict(img)
            discard_cards = cards_in_zone(detections, ZoneName.DEFAUSSE, calib_state['zone_map'])
            if discard_cards:
                top = discard_cards[0]
                print(f'  Top detected: {top.rank}{top.suit} (conf {top.confidence*100:.0f}%)')
            else:
                print('  ⚠ No card detected in DEFAUSSE.')

        # 6. End-of-turn check
        if counting.is_opponent_empty(1):
            print('🏁 RAMAI won!')
            voice.say('J\'ai gagné.')
            state.winner = 1
            state.terminal = True
        else:
            state.current = 0
            state.turn += 1
            voice.say('À toi.')
            print(f'→ Your turn. Top discard: {state.top_discard.name}')

ai_play_btn.on_click(on_ai_play)

## Cell 8 — Cheat button (reveal AI hand)

In [ ]:
triche_btn = widgets.Button(description='👁 Cheat: reveal RAMAI hand', button_style='danger')
triche_out = widgets.Output()
display(triche_btn, triche_out)
triche_confirmed = [False]

def on_triche(b):
    triche_out.clear_output()
    if not triche_confirmed[0]:
        triche_confirmed[0] = True
        with triche_out:
            print('⚠ Cheat warning. Click again to confirm.')
        return
    triche_confirmed[0] = False
    with triche_out:
        if state.terminal:
            print('Game over.')
            return
        print('⚠ CHEAT — RAMAI hand:')
        print('  ', ' '.join(c.name for c in state.players[1].hand.cards))

triche_btn.on_click(on_triche)

## Cell 9 — Voice log (audit trail)

In [ ]:
print('=== VOICE LOG (audit trail) ===')
print()
history = voice.history()
print(f'Total phrases spoken: {len(history)}')
print()
for i, h in enumerate(history):
    print(f'{i+1:3d}. [{h["datetime"]}] {h["text"]}')
    print(f'     method: {h["method"]}, success: {h["success"]}')

## Cell 10 — End of game + stop stream

In [ ]:
print('=' * 60)
print('GAME ANALYSIS')
print('=' * 60)
print()
if state.winner is not None:
    winner = 'You' if state.winner == 0 else 'RAMAI'
    print(f'Winner: {winner}')
else:
    print('Game not finished.')
print(f'Turns played: {state.turn}')
print(f'Phrases spoken: {len(voice.history())}')
print()
print('Stopping stream...')
stop_stream()
voice.say('Partie terminée. Au revoir.')
print('✓ Stream stopped.')

## Cell 11 — Run tests

In [ ]:
!cd {REPO} && python -m pytest tests/ 2>&1 | tail -5